In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
GroqApiKey = os.getenv("GroqAPI")
ModelName = os.getenv("ModelName")

In [ ]:
os.environ["LangChainTracingV2"] = os.getenv( "LangChainTracingV2" )
os.environ["LangChainAPI"] = os.getenv( "LangChainAPI" )
os.environ["LangChainProject"] = os.getenv( "LangChainProject" )

In [ ]:
from langchain_groq import ChatGroq

LLM = ChatGroq(
    model=ModelName,
    api_key=GroqApiKey
)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

Loader = PyPDFLoader( "data/CourseNotes.pdf" )
Documents = Loader.load()

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

TextSplitter = RecursiveCharacterTextSplitter( chunk_size=1000, chunk_overlap=200 )
Chunks = TextSplitter.split_documents(Documents )

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
EmbeddingModel = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2" )

In [ ]:
from langchain_chroma import Chroma

VectorStore = Chroma.from_documents( documents = Chunks, embedding = EmbeddingModel, persist_directory = "chroma_db" )

In [ ]:
Retriever = VectorStore.as_retriever( search_kwargs = { "k": 4 } )

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):

    Question: str
    RetrievedDocs: list
    Answer: str
    Decision: str
    Sources: list

In [ ]:
from utils.InputGuardrail import detectPromptInjection, isOffTopic

def inputGuardrailNode(State):

    Question = State["Question"]

    if detectPromptInjection(Question):
        State["Decision"] = "BLOCK"
        return State

    if isOffTopic(Question,VectorStore):
        State["Decision"] = "BLOCK"
        return State

    RetrievedDocs = Retriever.invoke(Question)
    State["Decision"] = "PASS"
    State["RetrievedDocs"] = (RetrievedDocs )

    return State

In [ ]:
def rejectionNode(State):
    State["Answer"] = ( "Sorry, your request was blocked because it is either off-topic or contains prompt injection attempts.")

    return State

In [ ]:
def generationNode(State):

    Context = "\n\n".join( [ Doc.page_content for Doc in State["RetrievedDocs"] ])

    Prompt = f"""
Answer using only the context.
Context:
{Context}
Question:
{State["Question"]}

"""

    Response = LLM.invoke(Prompt)

    State["Answer"] = (Response.content)

    return State

In [ ]:
from utils.OutputGuardrail import detectHallucination

def outputGuardrailNode(State):

    Flagged = detectHallucination(State["Answer"], State["RetrievedDocs"])

    if Flagged:
        State["Decision"] = "FLAG"
        State["Answer"] = ( "[WARNING: POSSIBLE HALLUCINATION]\n\n" + State["Answer"])

    return State

In [ ]:
def sourceNode(State):
    State["Sources"] = []

    for Doc in State["RetrievedDocs"]:
        Source = Doc.metadata.get( "source", "Unknown" )

        State["Sources"].append(Source)

    return State

In [ ]:
def routeDecision(State):
    if State["Decision"] == "BLOCK":
        return "reject"

    return "generate"

In [ ]:
from langgraph.graph import StateGraph, END

Workflow = StateGraph(AgentState)
Workflow.add_node("guardrail", inputGuardrailNode)
Workflow.add_node("reject", rejectionNode)
Workflow.add_node("generate", generationNode)
Workflow.add_node("output_guardrail", outputGuardrailNode)
Workflow.add_node("sources", sourceNode)

In [ ]:
Workflow.set_entry_point("guardrail")
Workflow.add_conditional_edges( "guardrail", routeDecision, { "reject": "reject", "generate": "generate"} )
Workflow.add_edge( "generate", "output_guardrail" )
Workflow.add_edge( "output_guardrail", "sources" )
Workflow.add_edge( "sources", END )
Workflow.add_edge( "reject", END )

In [ ]:
Graph = Workflow.compile()

In [ ]:
Question = input("Ask Question: " )

In [ ]:
Result = Graph.invoke( {"Question": Question}, config={"run_name": "Guardrailed_RAG_Session", "tags": ["rag","guardrails","langgraph"], "metadata": {"model": ModelName} } )

In [ ]:
from utils.ReportGenerator import addSessionRecord, saveSessionReport

addSessionRecord(
    Question = Question,
    Decision = Result["Decision"],
    Answer = Result["Answer"],
    Sources = Result.get("Sources", []) )

ReportPath = saveSessionReport()

In [ ]:
print("\nDecision:", Result["Decision"] )

print("\nAnswer:\n")

print(Result["Answer"])

print("\nSources:")

for Source in Result.get("Sources", []):
    print(Source)

print("\nReport Saved:")
print(ReportPath)